# Import Libraries

In [ ]:
%pip install torch transformers numpy tqdm pandas ipywidgets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.9 MB/s eta 0:00:00


In [ ]:
import os

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

from dataset import *
from model import *
from trainer import Trainer

torch.manual_seed(42)

In [ ]:
class Config:
    MODEL_NAME = "cointegrated/rubert-tiny2"
    MODEL_TYPE = "custom"

    MAX_LEN = 128
    BATCH_SIZE = 64

    NUM_WORKERS = 2
    PIN_MEMORY = False
    CACHE_TOKENS = True

    N_EPOCHS = 7
    LR = 2e-5
    WEIGHT_DECAY = 1e-4
    WARMUP_RATIO = 0.1
    LABEL_SMOOTHING = 0.05
    PATIENCE = 3
    SCHEDULER = "cosine"
    LOSS_TYPE = "cross_entropy"  # или "focal"

    # Техники ускорения
    USE_AMP = False
    GRAD_ACCUM_STEPS = 1
    GRADIENT_CHECKPOINTING = False  # Экономит память, но медленнее

    # Для кастомной модели
    USE_ATTENTION = True  # Multi-head attention pooling
    NUM_ATTENTION_HEADS = 4
    DROPOUT_RATE = 0.2
    FREEZE_LAYERS = 0  # Заморозить первые N слоёв BERT (0 = все обучаются)

    DEVICE = "cpu"

# Loading data

In [ ]:
config = Config()
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

train_data.head()

,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


# Label encoding

In [ ]:
le = LabelEncoder()

train_data.rate = le.fit_transform(train_data.rate)
train_data.head()

,rate,text
0,3,Очень понравилось. Были в начале марта с соба...
1,4,В целом магазин устраивает.\nАссортимент позво...
2,4,"Очень хорошо что открылась 5 ка, теперь не над..."
3,2,Пятёрочка громко объявила о том как она заботи...
4,2,"Тесно, вечная сутолока, между рядами трудно ра..."


# Train Test split

In [ ]:
train_split, val_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=train_data['rate']
)

# Loading tokenizer from pretrained

In [ ]:
# Можно выбрать разные модели
tokenizer = AutoTokenizer.from_pretrained(
    "cointegrated/rubert-tiny2", truncation=True, do_lower_case=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# Creating datasets and dataloaders

In [ ]:
train_dataset = FiveDataset(
    train_split, tokenizer, config.MAX_LEN, cache_tokens=config.CACHE_TOKENS
)
val_dataset = FiveDataset(
    val_split, tokenizer, config.MAX_LEN, cache_tokens=config.CACHE_TOKENS
)
test_dataset = FiveDataset(
    test_data, tokenizer, config.MAX_LEN, cache_tokens=config.CACHE_TOKENS
)

Предварительное кэширование токенов...
Кэширование завершено. Объём: 41365 образцов
Предварительное кэширование токенов...
Кэширование завершено. Объём: 7300 образцов
Предварительное кэширование токенов...
Кэширование завершено. Объём: 12167 образцов


In [ ]:

loader_params = {
    'batch_size': config.BATCH_SIZE,
    'num_workers': config.NUM_WORKERS,
    'pin_memory': config.PIN_MEMORY,
    'persistent_workers': config.NUM_WORKERS > 0
}

train_loader = torch.utils.data.DataLoader(
    train_dataset, shuffle=True, collate_fn=FiveDataset.collate_fn, **loader_params
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, shuffle=False, collate_fn=FiveDataset.collate_fn, **loader_params
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, shuffle=False, collate_fn=FiveDataset.collate_fn, **loader_params
)

# Loading pretrained model from Huggingface

In [ ]:
model_config = {
    'num_classes': 5,
    'use_attention': config.USE_ATTENTION,
    'num_attention_heads': config.NUM_ATTENTION_HEADS,
    'dropout_rate': config.DROPOUT_RATE,
    'attention_dropout': 0.1,
    'freeze_layers': config.FREEZE_LAYERS,
    'gradient_checkpointing': config.GRADIENT_CHECKPOINTING,
    'use_fp16': config.USE_AMP
}
model = CustomBertForClassification(
    model_name=config.MODEL_NAME,
    config=model_config
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Creating Trainer object and fitting the model

In [ ]:
trainer_config = {
    'n_epochs': config.N_EPOCHS,
    'lr': config.LR,
    'weight_decay': config.WEIGHT_DECAY,
    'warmup_ratio': config.WARMUP_RATIO,
    'label_smoothing': config.LABEL_SMOOTHING,
    'patience': config.PATIENCE,
    'scheduler': config.SCHEDULER,
    'loss_type': config.LOSS_TYPE,
    'use_amp': config.USE_AMP,
    'grad_accum_steps': config.GRAD_ACCUM_STEPS,
    'device': config.DEVICE,
    'verbose': True
}
t = Trainer(trainer_config)

In [ ]:
t.fit(model, train_loader, val_loader)


Эпоха 1/7
LR: 0.00e+00


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

Валидация:   0%|          | 0/115 [00:00<?, ?it/s]

Train Loss: 1.5703 | Train Acc: 0.4056 | Train F1: 0.4462
Val Loss: 1.1012 | Val Acc: 0.6271 | Val F1: 0.6091
✓ Новая лучшая модель! F1: 0.6091

Эпоха 2/7
LR: 1.99e-05


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

...

Train Loss: 0.9416 | Train Acc: 0.6694 | Train F1: 0.6542
Val Loss: 0.9555 | Val Acc: 0.6610 | Val F1: 0.6440

Эпоха 7/7
LR: 1.22e-06


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

Валидация:   0%|          | 0/115 [00:00<?, ?it/s]

Train Loss: 0.9383 | Train Acc: 0.6715 | Train F1: 0.6567
Val Loss: 0.9519 | Val Acc: 0.6612 | Val F1: 0.6438

Лучший F1: 0.6449


CustomBertForClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, el

In [ ]:
val_predictions = t.predict(val_loader)
val_f1 = f1_score(val_split['rate'].tolist(), val_predictions, average='weighted')


print(f"F1-score (weighted): {val_f1:.4f}")
print("\nClassification Report:")
print(classification_report(val_split['rate'].tolist(), val_predictions))

Предсказание:   0%|          | 0/115 [00:00<?, ?it/s]

F1-score (weighted): 0.6449

Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.63      0.59       621
           1       0.18      0.05      0.07       362
           2       0.41      0.44      0.43       919
           3       0.46      0.39      0.42      1488
           4       0.81      0.88      0.84      3910

    accuracy                           0.66      7300
   macro avg       0.48      0.48      0.47      7300
weighted avg       0.63      0.66      0.64      7300



# Get testset predictions


In [ ]:
test_predictions = t.predict(test_loader)

# Обратное кодирование
test_predictions = le.inverse_transform(test_predictions)

sample_submission = pd.read_csv("sample_submission.csv")
sample_submission["rate"] = test_predictions

print("\nРаспределение предсказаний на тесте:")
print(sample_submission["rate"].value_counts().sort_index())


sample_submission.to_csv("submission_transformer.csv", index=False)

Предсказание:   0%|          | 0/191 [00:00<?, ?it/s]


Распределение предсказаний на тесте:
rate
1    1211
2     151
3    1582
4    2115
5    7108
Name: count, dtype: int64
